# Parking Sign Detection - Experiment C: Controlled Negatives (20% ratio) + Augmentation ON

Tests whether a controlled dose of negative samples combined with augmentation is stable and beneficial.

**Hypothesis:** A small ratio (~20%) of negative samples combined with augmentation reduces false positives without causing the instability seen when 91% of the dataset was negatives.

**Variable under test:** Negative sample ratio (20% vs 0% in Version B vs 91% in original)
**Control:** Same augmentation as Version B (exp7_stabilized_v3)

**Dataset:** ~2,570 positive images + ~640 randomly sampled negatives (~20% ratio)
- Combined from: parking-sign-coco (Roboflow) + sf-parking-signs (Figure Eight)
- All images resized to 640x640
- **Controlled subset of negative background crops INCLUDED**
- Train/Val/Test split: 80/10/10

**Model:** YOLO11m (Medium)

**Baselines:**
- Version B (no negatives, aug ON): mAP50=0.989, mAP50-95=0.758, P=0.980, R=0.962
- Version A (all negatives, aug OFF): mAP50≈0.976, mAP50-95≈0.775

## 1. Setup

In [ ]:
# Fix numpy version first (torchvision requires numpy<2)
!pip uninstall numpy -y -q
!pip install "numpy<2" -q

# Then install ultralytics
!pip install ultralytics -q

# Restart runtime after this cell if needed

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt
import shutil
import yaml
import os

# Kaggle dataset path (nested directory structure confirmed by user)
DATASET_PATH = Path("/kaggle/input/parking-sign-detection-coco-dataset/parking-sign-detection-coco-dataset")
OUTPUT_PATH = Path("/kaggle/working")
DATA_YAML = DATASET_PATH / "data.yaml"

print(f"Dataset Path: {DATASET_PATH}")
print(f"Dataset exists: {DATASET_PATH.exists()}")

if DATASET_PATH.exists():
    print(f"Contents: {list(DATASET_PATH.iterdir())}")
    
    # Parse data.yaml
    with open(DATA_YAML) as f:
        data_config = yaml.safe_load(f)
    
    print("Original config:", data_config)
    
    # Update paths to use absolute paths
    new_config = data_config.copy()
    
    for split in ["train", "val", "test"]:
        keys = [split]
        if split == 'val': keys.append('valid')
        
        for k in keys:
            if k in new_config:
                folder_name = "valid" if split == "val" else split
                
                img_path = DATASET_PATH / folder_name / "images"
                if not img_path.exists():
                    img_path = DATASET_PATH / folder_name
                
                if img_path.exists():
                    new_config[k] = str(img_path)
                    print(f"Resolved {k} to {img_path}")
                else:
                    print(f"WARNING: Could not find image path for {k} at {img_path}")

    new_config['path'] = str(DATASET_PATH)
    
    FIXED_DATA_YAML = OUTPUT_PATH / "data.yaml"
    with open(FIXED_DATA_YAML, "w") as f:
        yaml.dump(new_config, f)
    
    DATA_YAML = FIXED_DATA_YAML
    print(f"\nFixed data.yaml written to {DATA_YAML}")
    !cat {DATA_YAML}
else:
    print("ERROR: DATASET_PATH not found. Please check directory structure in /kaggle/input")
    !ls -R /kaggle/input | head -n 20

## 2. Dataset Verification

In [ ]:
# Count images in each split
for split in ["train", "valid", "test"]:
    try:
        folder_name = split
        if split == 'val' and not (DATASET_PATH / split).exists():
             folder_name = 'valid'
             
        img_dir = DATASET_PATH / folder_name / "images"
        label_dir = DATASET_PATH / folder_name / "labels"
        
        if img_dir.exists():
            n_images = len(list(img_dir.glob("*.jpg")))
            n_labels = len(list(label_dir.glob("*.txt")))
            print(f"{split} ({folder_name}): {n_images} images, {n_labels} labels")
        else:
            print(f"{split}: Directory not found at {img_dir}")
    except Exception as e:
        print(f"Error checking {split}: {e}")

## 3. Build Controlled-Negatives Dataset

Keep all positive images (non-empty labels) and randomly sample negatives to achieve ~20% negative ratio in the training set.
Validation and test sets keep all images unchanged (including any negatives) for fair evaluation.

In [ ]:
import random
import shutil
from pathlib import Path

random.seed(42)

TARGET_NEGATIVE_RATIO = 0.20  # 20% of final training set should be negatives
FILTERED_PATH = OUTPUT_PATH / "dataset_controlled_negatives"

for split in ["train", "valid", "test"]:
    src_img_dir = DATASET_PATH / split / "images"
    src_lbl_dir = DATASET_PATH / split / "labels"
    dst_img_dir = FILTERED_PATH / split / "images"
    dst_lbl_dir = FILTERED_PATH / split / "labels"
    
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)
    
    if not src_img_dir.exists():
        print(f"Skipping {split}: {src_img_dir} not found")
        continue
    
    # Separate positives and negatives
    positives = []
    negatives = []
    
    for img_file in sorted(src_img_dir.glob("*")):
        lbl_file = src_lbl_dir / (img_file.stem + ".txt")
        is_negative = lbl_file.exists() and lbl_file.stat().st_size == 0
        if is_negative:
            negatives.append(img_file)
        else:
            positives.append(img_file)
    
    # For train split: sample negatives to achieve target ratio
    # For valid/test: keep everything for fair evaluation
    if split == "train":
        n_positives = len(positives)
        # target_neg / (n_positives + target_neg) = TARGET_NEGATIVE_RATIO
        # target_neg = TARGET_NEGATIVE_RATIO * n_positives / (1 - TARGET_NEGATIVE_RATIO)
        target_neg = int(TARGET_NEGATIVE_RATIO * n_positives / (1 - TARGET_NEGATIVE_RATIO))
        sampled_negatives = random.sample(negatives, min(target_neg, len(negatives)))
        keep = positives + sampled_negatives
        print(f"{split}: {len(positives)} positives + {len(sampled_negatives)} negatives "
              f"(sampled from {len(negatives)}) = {len(keep)} total "
              f"({len(sampled_negatives)/len(keep)*100:.1f}% negative)")
    else:
        keep = positives + negatives
        print(f"{split}: {len(positives)} positives + {len(negatives)} negatives = {len(keep)} total")
    
    # Copy selected images and labels
    for img_file in keep:
        shutil.copy2(img_file, dst_img_dir / img_file.name)
        lbl_file = src_lbl_dir / (img_file.stem + ".txt")
        if lbl_file.exists():
            shutil.copy2(lbl_file, dst_lbl_dir / lbl_file.name)

# Write filtered data.yaml
filtered_config = {
    'path': str(FILTERED_PATH),
    'train': str(FILTERED_PATH / 'train' / 'images'),
    'val': str(FILTERED_PATH / 'valid' / 'images'),
    'test': str(FILTERED_PATH / 'test' / 'images'),
    'nc': 1,
    'names': ['parking_sign'],
}

FILTERED_DATA_YAML = FILTERED_PATH / "data.yaml"
with open(FILTERED_DATA_YAML, 'w') as f:
    yaml.dump(filtered_config, f)

print(f"\nFiltered data.yaml written to {FILTERED_DATA_YAML}")
!cat {FILTERED_DATA_YAML}

## 4. Training Configuration

### Experiment C: Controlled Negatives (20%) + Augmentation ON

Uses the same augmentation as Version B (exp7_stabilized_v3) but reintroduces a controlled subset of negatives.

### Termination Strategy
- **Epoch ceiling:** 80 (generous upper bound)
- **Patience:** 8 epochs without mAP50 improvement → early stop
- **Hypothesis confirmed if:** Training is stable (no loss oscillation) AND mAP50 ≥ 0.985 AND precision improves over Version B's 0.980
- **Hypothesis disproved if:** Loss oscillates (instability returns) OR mAP50 plateaus significantly below 0.985
- **Key comparison:** Does the 20% negative dose improve precision without hurting recall?

In [ ]:
RUN_NAME = "parking_sign_controlled_neg"

TRAIN_PARAMS = {
    "data": str(FILTERED_DATA_YAML),
    "epochs": 80,
    "imgsz": 640,
    "batch": 16,
    "workers": 4,
    "patience": 8,
    "save_period": 10,
    "project": str(OUTPUT_PATH / "runs"),
    "name": RUN_NAME,
    "exist_ok": True,
    "pretrained": True,
    "verbose": True,
    # Multi-GPU (DDP)
    "device": "0,1",
    # Cosine LR decay
    "cos_lr": True,
    "lr0": 0.005,
    "lrf": 0.005,
    # Augmentation ON — same as Version B (exp7_stabilized_v3)
    "mosaic": 1.0,
    "mixup": 0.08,
    "copy_paste": 0.05,
    "degrees": 8.0,
    "translate": 0.1,
    "scale": 0.4,
    "shear": 2.0,
    "perspective": 0.0001,
    "fliplr": 0.5,
    "flipud": 0.0,
    "hsv_h": 0.01,
    "hsv_s": 0.5,
    "hsv_v": 0.3,
    "close_mosaic": 25,
}

print(f"Training config: {len(TRAIN_PARAMS)} parameters")
print(f"Model: yolo11m | Max epochs: {TRAIN_PARAMS['epochs']}, Patience: {TRAIN_PARAMS['patience']}")
print(f"Image size: {TRAIN_PARAMS['imgsz']} | Device: {TRAIN_PARAMS['device']} (2 GPUs)")
print(f"Negative annotations: CONTROLLED (~20%) | Augmentation: ON")
print(f"  mosaic={TRAIN_PARAMS['mosaic']}, scale={TRAIN_PARAMS['scale']}, mixup={TRAIN_PARAMS['mixup']}")
print(f"\nTermination: early stop after {TRAIN_PARAMS['patience']} epochs without mAP50 improvement")

## 5. Train Model

In [ ]:
print("="*60)
print("Training Experiment C: Controlled Negatives + Augmentation ON")
print("="*60)

model = YOLO("yolo11m.pt")
train_results = model.train(**TRAIN_PARAMS)

## 6. Training Results

In [ ]:
# Show training curves
from IPython.display import Image, display

results_img = OUTPUT_PATH / "runs" / RUN_NAME / "results.png"
if results_img.exists():
    print("Training curves:")
    display(Image(filename=str(results_img)))
else:
    print(f"Results image not found at {results_img}")

## 7. Evaluate Best Model

In [ ]:
# Load best model
best_model_path = OUTPUT_PATH / "runs" / RUN_NAME / "weights" / "best.pt"
best_model = YOLO(best_model_path)

print(f"Loaded model from: {best_model_path}")

# Run validation on test set
print("\nEvaluating on test set...")
test_results = best_model.val(data=str(FILTERED_DATA_YAML), split="test")

test_mAP50 = test_results.results_dict.get('metrics/mAP50(B)', 0)
test_mAP50_95 = test_results.results_dict.get('metrics/mAP50-95(B)', 0)
test_precision = test_results.results_dict.get('metrics/precision(B)', 0)
test_recall = test_results.results_dict.get('metrics/recall(B)', 0)

print(f"\nTest Set Results:")
print(f"  Precision:  {test_precision:.4f}")
print(f"  Recall:     {test_recall:.4f}")
print(f"  mAP50:      {test_mAP50:.4f}")
print(f"  mAP50-95:   {test_mAP50_95:.4f}")

# Compare with baselines
print(f"\nComparison with Version B (no negatives, aug ON):")
print(f"  Version B  P=0.980, R=0.962, mAP50=0.989, mAP50-95=0.758")
print(f"  This run   P={test_precision:.3f}, R={test_recall:.3f}, mAP50={test_mAP50:.3f}, mAP50-95={test_mAP50_95:.3f}")
print(f"  Precision delta: {test_precision - 0.980:+.4f}")
print(f"  Recall delta:    {test_recall - 0.962:+.4f}")

if test_mAP50 >= 0.985 and test_precision > 0.980:
    print("  \u2192 HYPOTHESIS CONFIRMED: Controlled negatives improve precision without instability")
elif test_mAP50 >= 0.985:
    print("  \u2192 PARTIAL: Stable training but no precision improvement from negatives")
else:
    print("  \u2192 HYPOTHESIS NOT CONFIRMED: Controlled negatives did not help")

In [ ]:
# Show confusion matrix
confusion_img = OUTPUT_PATH / "runs" / RUN_NAME / "confusion_matrix.png"
if confusion_img.exists():
    print("Confusion matrix:")
    display(Image(filename=str(confusion_img)))

In [ ]:
# === Additional diagnostics ===
import numpy as np

# 1. Detailed per-image predictions on test set
print("Running detailed test set analysis...")
test_preds = best_model.predict(
    source=str(FILTERED_PATH / "test" / "images"),
    conf=0.01,
    save_txt=True,
    save_conf=True,
    project=str(OUTPUT_PATH / "runs"),
    name=f"{RUN_NAME}_test_predictions",
    exist_ok=True,
)

# 2. Confidence distribution
all_confs = []
for r in test_preds:
    if r.boxes is not None and len(r.boxes):
        all_confs.extend(r.boxes.conf.cpu().numpy().tolist())

if all_confs:
    confs = np.array(all_confs)
    print(f"\nConfidence distribution (test set, conf>0.01):")
    print(f"  N predictions: {len(confs)}")
    print(f"  Mean: {confs.mean():.3f}, Median: {np.median(confs):.3f}")
    print(f"  <0.25: {(confs < 0.25).sum()}, 0.25-0.5: {((confs >= 0.25) & (confs < 0.5)).sum()}")
    print(f"  0.5-0.75: {((confs >= 0.5) & (confs < 0.75)).sum()}, >0.75: {(confs >= 0.75).sum()}")

print(f"\nDiagnostic outputs saved to: {OUTPUT_PATH / 'runs' / f'{RUN_NAME}_test_predictions'}")

## 8. Export Best Model

In [ ]:
# Export to ONNX for deployment
print("Exporting best model to ONNX...")
onnx_path = best_model.export(format="onnx", imgsz=640, simplify=True)
print(f"Exported to: {onnx_path}")

# Also save PyTorch format to output root
final_model_path = OUTPUT_PATH / f"parking_sign_detector_{RUN_NAME}.pt"
shutil.copy(best_model_path, final_model_path)
print(f"PyTorch model: {final_model_path}")

## 9. Results Summary

### Experiment C: Controlled Negatives (20%) + Augmentation ON
- **Max epochs:** 80, patience 8 (early stop on mAP50 plateau)
- **Negative annotations:** CONTROLLED (~20% of training set, randomly sampled)
- **Augmentation:** ON (same as Version B: mosaic=1.0, scale=0.4, mixup=0.08, etc.)
- **Hypothesis:** A small negative dose reduces FP (improves precision) without causing instability

### Decision Criteria
| Outcome | Interpretation |
|---|---|
| Stable + P > 0.980 + mAP50 ≥ 0.985 | ✅ Controlled negatives are beneficial |
| Stable + P ≈ 0.980 + mAP50 ≥ 0.985 | ↔ Negatives neutral at this ratio |
| Unstable (oscillating loss) | ❌ Even 20% negatives + aug causes instability |
| mAP50 < 0.985 | ❌ Negatives hurt overall detection quality |

In [ ]:
# Print final summary
print("\n" + "="*60)
print("EXPERIMENT C (CONTROLLED NEGATIVES + AUG ON) COMPLETE")
print("="*60)
print(f"\nRun name: {RUN_NAME}")
print(f"Output files:")
print(f"  - {final_model_path}")
print(f"  - {OUTPUT_PATH / 'runs' / RUN_NAME / 'results.csv'}")